In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load files
df = pd.read_csv("T_ONTIME_REPORTING.csv", low_memory=False)
doc = pd.read_csv("Documentation.csv")

# Basic shapes
print("Dataset shape:", df.shape)
print("Documentation shape:", doc.shape)

# First rows
print("\nDataset preview:")
print(df.head())

print("\nDocumentation preview:")
print(doc.head())

Dataset shape: (539747, 109)
Documentation shape: (109, 2)

Dataset preview:
   YEAR  QUARTER  MONTH  DAY_OF_MONTH  DAY_OF_WEEK               FL_DATE  \
0  2025        1      1             1            3  1/1/2025 12:00:00 AM   
1  2025        1      1             1            3  1/1/2025 12:00:00 AM   
2  2025        1      1             1            3  1/1/2025 12:00:00 AM   
3  2025        1      1             1            3  1/1/2025 12:00:00 AM   
4  2025        1      1             1            3  1/1/2025 12:00:00 AM   

  OP_UNIQUE_CARRIER  OP_CARRIER_AIRLINE_ID OP_CARRIER TAIL_NUM  ...  \
0                AA                  19805         AA   N101NN  ...   
1                AA                  19805         AA   N101NN  ...   
2                AA                  19805         AA   N102UW  ...   
3                AA                  19805         AA   N103NN  ...   
4                AA                  19805         AA   N103NN  ...   

   DIV4_WHEELS_OFF  DIV4_TAIL_NUM  DIV5

In [3]:
field_descriptions = dict(zip(doc["SYS_FIELD_NAME"], doc["FIELD_DESC"]))

In [4]:
print(field_descriptions['ARR_DELAY'])

Difference in minutes between scheduled and actual arrival time. Early arrivals show negative numbers.


In [5]:
summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_values": df.isnull().sum().values,
    "missing_pct": (df.isnull().mean() * 100).round(2).values,
    "description": [field_descriptions.get(col, "No description found") for col in df.columns]
})

print(summary.head(50))

                   column    dtype  missing_values  missing_pct  \
0                    YEAR    int64               0         0.00   
1                 QUARTER    int64               0         0.00   
2                   MONTH    int64               0         0.00   
3            DAY_OF_MONTH    int64               0         0.00   
4             DAY_OF_WEEK    int64               0         0.00   
5                 FL_DATE   object               0         0.00   
6       OP_UNIQUE_CARRIER   object               0         0.00   
7   OP_CARRIER_AIRLINE_ID    int64               0         0.00   
8              OP_CARRIER   object               0         0.00   
9                TAIL_NUM   object            2530         0.47   
10      OP_CARRIER_FL_NUM    int64               0         0.00   
11      ORIGIN_AIRPORT_ID    int64               0         0.00   
12  ORIGIN_AIRPORT_SEQ_ID    int64               0         0.00   
13  ORIGIN_CITY_MARKET_ID    int64               0         0.0

In [6]:
print("Rows, columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values (top 30):")
print(df.isnull().sum().sort_values(ascending=False).head(30))

print("\nDuplicate rows:")
print(df.duplicated().sum())

Rows, columns: (539747, 109)

Column names:
['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_FIPS', 'ORIGIN_STATE_NM', 'ORIGIN_WAC', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID', 'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_FIPS', 'DEST_STATE_NM', 'DEST_WAC', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'DEP_DELAY_GROUP', 'DEP_TIME_BLK', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'ARR_DELAY_GROUP', 'ARR_TIME_BLK', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'FLIGHTS', 'DISTANCE', 'DISTANCE_GROUP', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY

In [7]:
# Build a clean overview table
overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().mean() * 100).round(2),
    "description": [field_descriptions.get(col, "No description found") for col in df.columns]
}).sort_values(by="missing_pct", ascending=False)

print(overview)

                                column    dtype  missing_count  missing_pct  \
DIV5_TAIL_NUM            DIV5_TAIL_NUM  float64         539747        100.0   
DIV3_AIRPORT              DIV3_AIRPORT  float64         539747        100.0   
DIV3_WHEELS_OFF        DIV3_WHEELS_OFF  float64         539747        100.0   
DIV3_LONGEST_GTIME  DIV3_LONGEST_GTIME  float64         539747        100.0   
DIV3_TOTAL_GTIME      DIV3_TOTAL_GTIME  float64         539747        100.0   
...                                ...      ...            ...          ...   
DEST_STATE_FIPS        DEST_STATE_FIPS    int64              0          0.0   
DEST_STATE_ABR          DEST_STATE_ABR   object              0          0.0   
DEST_CITY_NAME          DEST_CITY_NAME   object              0          0.0   
DEST                              DEST   object              0          0.0   
DISTANCE                      DISTANCE  float64              0          0.0   

                                                   

In [8]:
delay_cols = [col for col in df.columns if "DELAY" in col.upper() or "LATE" in col.upper()]

delay_info = pd.DataFrame({
    "column": delay_cols,
    "description": [field_descriptions.get(col, "No description found") for col in delay_cols],
    "missing_count": [df[col].isnull().sum() for col in delay_cols],
    "missing_pct": [round(df[col].isnull().mean() * 100, 2) for col in delay_cols]
})

print(delay_info)

                 column                                        description  \
0             DEP_DELAY  Difference in minutes between scheduled and ac...   
1         DEP_DELAY_NEW  Difference in minutes between scheduled and ac...   
2       DEP_DELAY_GROUP  Departure Delay intervals, every (15 minutes f...   
3             ARR_DELAY  Difference in minutes between scheduled and ac...   
4         ARR_DELAY_NEW  Difference in minutes between scheduled and ac...   
5       ARR_DELAY_GROUP  Arrival Delay intervals, every (15-minutes fro...   
6         CARRIER_DELAY                          Carrier Delay, in Minutes   
7         WEATHER_DELAY                          Weather Delay, in Minutes   
8             NAS_DELAY              National Air System Delay, in Minutes   
9        SECURITY_DELAY                         Security Delay, in Minutes   
10  LATE_AIRCRAFT_DELAY                    Late Aircraft Delay, in Minutes   
11        DIV_ARR_DELAY  Difference in minutes between scheduled

In [9]:
df["is_delayed"] = (df["ARR_DELAY"] > 15).astype(int)

In [10]:
leakage_cols = [
    "ARR_DELAY", "ARR_DELAY_NEW", "ARR_DELAY_GROUP",
    "DEP_DELAY", "DEP_DELAY_NEW", "DEP_DELAY_GROUP",
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
    "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
    "DIV_ARR_DELAY"
]

#df = df.drop(columns=leakage_cols)

In [11]:
df["is_delayed"].value_counts(normalize=True)

is_delayed
0    0.824318
1    0.175682
Name: proportion, dtype: float64

In [12]:
df = df.sort_values(by=["FL_DATE", "CRS_ARR_TIME"])

In [13]:
df["avg_delay_so_far"] = (
    df.groupby("FL_DATE")["ARR_DELAY"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

In [14]:
df["avg_delay_so_far"]

9               NaN
1531      -7.000000
7343       0.500000
6389      -1.666667
8518      -0.750000
            ...    
158432    10.156164
158588    10.155024
158597    10.154701
158736    10.152683
162079    10.153740
Name: avg_delay_so_far, Length: 539747, dtype: float64

In [15]:
global_mean = df["ARR_DELAY"].mean()
df["avg_delay_so_far"] = df["avg_delay_so_far"].fillna(global_mean)

In [16]:
df["is_delayed"] = (df["ARR_DELAY"] > 15).astype(int)

In [17]:
df["baseline_pred"] = (df["avg_delay_so_far"] > 15).astype(int)

In [18]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(df["is_delayed"], df["baseline_pred"]))
print(classification_report(df["is_delayed"], df["baseline_pred"]))

[[402477  42446]
 [ 76328  18496]]
              precision    recall  f1-score   support

           0       0.84      0.90      0.87    444923
           1       0.30      0.20      0.24     94824

    accuracy                           0.78    539747
   macro avg       0.57      0.55      0.55    539747
weighted avg       0.75      0.78      0.76    539747



In [19]:
df[["FL_DATE", "CRS_ARR_TIME", "ARR_DELAY", "avg_delay_so_far", "baseline_pred"]].head(15)

,FL_DATE,CRS_ARR_TIME,ARR_DELAY,avg_delay_so_far,baseline_pred
9,1/1/2025 12:00:00 AM,1,-7.0,3.756614,0
1531,1/1/2025 12:00:00 AM,1,8.0,-7.000000,0
7343,1/1/2025 12:00:00 AM,1,-6.0,0.500000,0
6389,1/1/2025 12:00:00 AM,2,2.0,-1.666667,0
8518,1/1/2025 12:00:00 AM,2,-1.0,-0.750000,0
1627,1/1/2025 12:00:00 AM,3,-18.0,-0.800000,0
1933,1/1/2025 12:00:00 AM,3,-15.0,-3.666667,0
2387,1/1/2025 12:00:00 AM,4,-7.0,-5.285714,0
8472,1/1/2025 12:00:00 AM,4,-17.0,-5.500000,0
1622,1/1/2025 12:00:00 AM,5,-5.0,-6.777778,0


In [20]:
df["avg_delay_airport"] = (
    df.groupby(["FL_DATE", "ORIGIN"])["ARR_DELAY"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=[0,1], drop=True)
)

In [21]:
df["avg_delay_airport"] = df["avg_delay_airport"].fillna(global_mean)

df["baseline_pred_airport"] = (df["avg_delay_airport"] > 5).astype(int)

In [22]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(df["is_delayed"], df["baseline_pred_airport"]))
print(classification_report(df["is_delayed"], df["baseline_pred_airport"]))

[[345883  99040]
 [ 48741  46083]]
              precision    recall  f1-score   support

           0       0.88      0.78      0.82    444923
           1       0.32      0.49      0.38     94824

    accuracy                           0.73    539747
   macro avg       0.60      0.63      0.60    539747
weighted avg       0.78      0.73      0.75    539747



In [23]:
df[["FL_DATE", "CRS_ARR_TIME", "ARR_DELAY","ORIGIN", "avg_delay_airport", "baseline_pred_airport"]].head(100)

,FL_DATE,CRS_ARR_TIME,ARR_DELAY,ORIGIN,avg_delay_airport,baseline_pred_airport
9,1/1/2025 12:00:00 AM,1,-7.0,PHX,0.342222,0
1531,1/1/2025 12:00:00 AM,1,8.0,DFW,4.564516,0
7343,1/1/2025 12:00:00 AM,1,-6.0,CLT,-14.666667,0
6389,1/1/2025 12:00:00 AM,2,2.0,CLT,-6.000000,0
8518,1/1/2025 12:00:00 AM,2,-1.0,CLT,-2.000000,0
...,...,...,...,...,...,...
5015,1/1/2025 12:00:00 AM,22,19.0,DTW,16.000000,1
5094,1/1/2025 12:00:00 AM,22,18.0,ATL,0.000000,0
5982,1/1/2025 12:00:00 AM,22,5.0,DEN,10.000000,1
2265,1/1/2025 12:00:00 AM,23,18.0,MIA,5.466667,1
